# 03 - Creación del pipeline de ingeniería de características

**Proyecto 2 - Pipeline de Machine Learning (Predict Future Sales)**

Brady Monroy (15003165) y Luis Rodolfo Rodriguez

En este notebook se construye el **pipeline de scikit-learn** con los pasos elegidos en el notebook 02 y se guarda
como archivo `.pkl`. Igual que en clase, el pipeline usa transformadores de `feature-engine` y se ajusta **solo con
el set de entrenamiento**.

Pasos del pipeline, en orden:
1. **Mapeo** de nombres del ruso al español (Mapper).
2. **Imputación** de variables numéricas y categóricas.
3. **Codificación** de variables categóricas (One-Hot).
4. **Tratamiento de outliers** (Winsorizer).
5. **Transformación** de variables numéricas (Yeo-Johnson).
6. **Escalado** (StandardScaler).

## Configuración

La lista de variables y los grupos de cada transformación se leen de `pipeline_config.yaml`, en la raíz del
proyecto (igual que en clase).

El **Mapper** está en `src/features/preprocesadores.py`. Se guarda en un archivo `.py`, y no dentro del notebook,
para que el `.pkl` se pueda abrir después en los notebooks 04 y 05.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.encoding import OneHotEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)

# Rutas del proyecto
PROJ_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJ_ROOT / "data" / "processed"
MODELS_DIR = PROJ_ROOT / "models"

# Para poder importar el Mapper desde la carpeta src
sys.path.append(str(PROJ_ROOT))
from src.features.preprocesadores import Mapper

RANDOM_STATE = 42

with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

MODEL_FEATURES = config["MODEL_FEATURES"]
TARGET = config["TARGET"]
print("Variables del modelo:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

Variables del modelo: 10
['lag_1', 'lag_2', 'lag_3', 'ciudad', 'tienda_online', 'venta_tienda_lag_1', 'grupo_categoria', 'venta_producto_lag_1', 'meses_desde_primera_venta', 'mes']


## Carga de datos

Se carga la tabla mensual creada en el notebook 02 y se divide **en orden de tiempo**:
- Entrenamiento: meses 0 a 26.
- Validación: meses 27 a 33.
- Test: mes 34 (los pares de `test.csv`).

In [2]:
mensual = pd.read_parquet(PROCESSED_DATA_DIR / "ventas_mensuales.parquet")

es_train = (mensual["date_block_num"] >= config["PRIMER_MES_TRAIN"]) & (mensual["date_block_num"] <= config["ULTIMO_MES_TRAIN"])
es_valid = (mensual["date_block_num"] >= config["PRIMER_MES_VALID"]) & (mensual["date_block_num"] <= config["ULTIMO_MES_VALID"])
es_test = mensual["date_block_num"] == config["MES_TEST"]

X_train = mensual.loc[es_train, MODEL_FEATURES]
y_train = mensual.loc[es_train, TARGET]

X_valid = mensual.loc[es_valid, MODEL_FEATURES]
y_valid = mensual.loc[es_valid, TARGET]

X_test = mensual.loc[es_test, MODEL_FEATURES]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

X_train: (9299486, 10)
X_valid: (1614364, 10)
X_test : (214200, 10)


## Paso 1: Mapeo de nombres (Mapper)

Los nombres de las ciudades y de los grupos de categoría vienen en ruso. El **Mapper** los cambia al español usando
un diccionario. Si aparece un valor que no está en el diccionario, se pone `Otro`. Los valores vacíos se dejan vacíos
para que los rellene el paso de imputación.

In [3]:
# Diccionario de ciudades (primera palabra del nombre de la tienda)
CIUDADES = {
    "Москва": "Moscu", "Якутск": "Yakutsk", "Воронеж": "Voronezh", "РостовНаДону": "Rostov del Don",
    "Тюмень": "Tiumen", "Жуковский": "Zhukovski", "Казань": "Kazan", "Красноярск": "Krasnoyarsk",
    "Н.Новгород": "Nizhni Novgorod", "Новосибирск": "Novosibirsk", "СПб": "San Petersburgo",
    "Самара": "Samara", "Уфа": "Ufa", "Адыгея": "Adigueya", "Балашиха": "Balashija",
    "Волжский": "Volzhski", "Вологда": "Vologda", "Калуга": "Kaluga", "Коломна": "Kolomna",
    "Курск": "Kursk", "Мытищи": "Mytishchi", "Омск": "Omsk", "Сергиев": "Sergiev Posad",
    "Сургут": "Surgut", "Томск": "Tomsk", "Химки": "Jimki", "Чехов": "Chejov", "Ярославль": "Yaroslavl",
    # Estas tres no son ciudades
    "Выездная": "Ventas fuera de tienda", "Интернет-магазин": "Tienda en linea", "Цифровой": "Almacen digital",
}

# Diccionario de grupos de categoría (texto antes de " - " en el nombre de la categoría)
GRUPOS = {
    "Книги": "Libros", "Подарки": "Regalos", "Игровые консоли": "Consolas", "Игры": "Juegos de consola",
    "Аксессуары": "Accesorios", "Музыка": "Musica", "Программы": "Programas", "Кино": "Cine",
    "Игры PC": "Juegos de PC", "Карты оплаты": "Tarjetas de pago", "Служебные": "Servicio interno",
    "PC": "Audifonos de PC", "Билеты (Цифра)": "Boletos digitales", "Доставка товара": "Envios",
    "Игры Android": "Juegos Android", "Игры MAC": "Juegos MAC",
    "Карты оплаты (Кино, Музыка, Игры)": "Tarjetas de pago (cine, musica, juegos)",
    "Чистые носители (шпиль)": "Discos virgenes (paquete)", "Чистые носители (штучные)": "Discos virgenes (unidad)",
    "Элементы питания": "Baterias",
}

mapeo = [
    ("traducir_nombres", Mapper(mappings={"ciudad": CIUDADES, "grupo_categoria": GRUPOS})),
]

X_train_1 = Pipeline(mapeo).fit_transform(X_train)

print("Ciudades después del mapeo:", sorted(X_train_1["ciudad"].unique()))
print()
print("Grupos después del mapeo:", sorted(X_train_1["grupo_categoria"].unique()))
print()
print("Valores 'Otro' (no estaban en el diccionario):",
      (X_train_1[["ciudad", "grupo_categoria"]] == "Otro").sum().sum())

Ciudades después del mapeo: ['Adigueya', 'Almacen digital', 'Balashija', 'Chejov', 'Jimki', 'Kaluga', 'Kazan', 'Kolomna', 'Krasnoyarsk', 'Kursk', 'Moscu', 'Mytishchi', 'Nizhni Novgorod', 'Novosibirsk', 'Omsk', 'Rostov del Don', 'Samara', 'San Petersburgo', 'Sergiev Posad', 'Surgut', 'Tienda en linea', 'Tiumen', 'Tomsk', 'Ufa', 'Ventas fuera de tienda', 'Vologda', 'Volzhski', 'Voronezh', 'Yakutsk', 'Yaroslavl', 'Zhukovski']



Grupos después del mapeo: ['Accesorios', 'Audifonos de PC', 'Baterias', 'Boletos digitales', 'Cine', 'Consolas', 'Discos virgenes (paquete)', 'Discos virgenes (unidad)', 'Envios', 'Juegos Android', 'Juegos MAC', 'Juegos de PC', 'Juegos de consola', 'Libros', 'Musica', 'Programas', 'Regalos', 'Servicio interno', 'Tarjetas de pago', 'Tarjetas de pago (cine, musica, juegos)']

Valores 'Otro' (no estaban en el diccionario): 0


## Paso 2: Imputación

Según el notebook 02 (regla de clase por porcentaje de faltantes):
- `venta_tienda_lag_1` (4.88% de faltantes): **mediana**.
- `precio_producto_lag_1` (21.76% de faltantes) se **descartó** en el notebook 02, así que no entra al pipeline.
- `ciudad` y `grupo_categoria`: **moda**. No tienen vacíos en los datos reales, pero el paso queda en el pipeline por
  si llegan datos nuevos con vacíos. En el notebook 02 se forzaron vacíos para probarlo.

In [4]:
imputacion = [
    ("imputar_mediana", MeanMedianImputer(imputation_method="median",
                                          variables=config["IMPUTAR_MEDIANA"])),
    ("imputar_moda", CategoricalImputer(imputation_method="frequent",
                                        variables=config["CATEGORICAS"])),
]

X_train_2 = Pipeline(imputacion).fit_transform(X_train_1)

print("Vacíos antes :", X_train_1.isnull().sum().sum())
print("Vacíos después:", X_train_2.isnull().sum().sum())

Vacíos antes : 453373
Vacíos después: 0


## Paso 3: Codificación de variables categóricas

**One-Hot** en `ciudad` y `grupo_categoria`: una columna 0/1 por cada valor. Una categoría que no aparezca en
entrenamiento queda con todas sus columnas en 0.

In [5]:
codificacion = [
    ("one_hot", OneHotEncoder(variables=config["CATEGORICAS"])),
]

X_train_3 = Pipeline(codificacion).fit_transform(X_train_2)

print("Columnas:", X_train_2.shape[1], "->", X_train_3.shape[1])

Columnas: 10 -> 59


## Paso 4: Tratamiento de outliers

**Winsorizer** al percentil 99 en `venta_producto_lag_1`: los valores por encima del límite se recortan al límite.

In [6]:
outliers = [
    ("capping_p99", Winsorizer(capping_method="quantiles", tail="right", fold=0.01,
                               variables=config["OUTLIERS_P99"])),
]

pipe_outliers = Pipeline(outliers)
X_train_4 = pipe_outliers.fit_transform(X_train_3)

print("Límite aprendido:", pipe_outliers.named_steps["capping_p99"].right_tail_caps_)
print("Filas recortadas:", (X_train_3["venta_producto_lag_1"] != X_train_4["venta_producto_lag_1"]).sum())

Límite aprendido: {'venta_producto_lag_1': 3.82608699798584}
Filas recortadas: 92698


## Paso 5: Transformación de variables numéricas

**Yeo-Johnson** en las variables con sesgo mayor a 0.75 (elegidas en el notebook 02).

In [7]:
transformacion = [
    ("yeo_johnson", YeoJohnsonTransformer(variables=config["YEO_JOHNSON"])),
]

pipe_transformacion = Pipeline(transformacion)
X_train_5 = pipe_transformacion.fit_transform(X_train_4)

pd.DataFrame({
    "sesgo_antes": X_train_4[config["YEO_JOHNSON"]].skew(),
    "sesgo_despues": X_train_5[config["YEO_JOHNSON"]].skew(),
    "lambda": pd.Series(pipe_transformacion.named_steps["yeo_johnson"].lambda_dict_),
}).round(3)

,sesgo_antes,sesgo_despues,lambda
lag_1,9.479,2.107,-5.956
lag_2,9.684,2.200,-5.956
lag_3,9.962,2.310,-5.956
venta_tienda_lag_1,2.302,0.098,-4.160
venta_producto_lag_1,4.336,0.619,-4.186


## Paso 6: Escalado

**StandardScaler** (media 0 y desviación 1) en las variables numéricas. Las columnas One-Hot y `tienda_online` no se
escalan porque ya son 0/1.

In [8]:
escalado = [
    ("escalado_estandar", SklearnTransformerWrapper(transformer=StandardScaler(),
                                                    variables=config["ESCALAR"])),
]

X_train_6 = Pipeline(escalado).fit_transform(X_train_5)

X_train_6[config["ESCALAR"]].describe().loc[["mean", "std"]].round(2)

,lag_1,lag_2,lag_3,venta_tienda_lag_1,venta_producto_lag_1,mes,meses_desde_primera_venta
mean,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Pipeline final

Se unen los seis pasos en un solo `Pipeline`, se ajusta con `X_train` y se verifica, igual que en clase, que da
**exactamente el mismo resultado** que aplicar los pasos uno por uno. También se revisa que no queden vacíos en
ningún set y que los tres sets tengan las mismas columnas.

In [9]:
# Borrar las tablas intermedias para liberar memoria (solo se necesita X_train_6 para comparar)
del X_train_1, X_train_2, X_train_3, X_train_4, X_train_5

pipeline_fe = Pipeline(mapeo + imputacion + codificacion + outliers + transformacion + escalado)
pipeline_fe.fit(X_train)

X_train_fe = pipeline_fe.transform(X_train)
X_valid_fe = pipeline_fe.transform(X_valid)
X_test_fe = pipeline_fe.transform(X_test)

# Verificaciones
# 1) Mismo resultado que los pasos uno por uno (se compara columna por columna para no llenar la memoria)
for columna in X_train_fe.columns:
    assert np.allclose(X_train_fe[columna].astype(float), X_train_6[columna].astype(float))

# 2) Sin vacíos y con las mismas columnas en los tres sets
assert X_train_fe.isnull().sum().sum() == 0
assert X_valid_fe.isnull().sum().sum() == 0
assert X_test_fe.isnull().sum().sum() == 0
assert list(X_train_fe.columns) == list(X_valid_fe.columns) == list(X_test_fe.columns)

print("Todas las verificaciones pasaron")
print("X_train_fe:", X_train_fe.shape)
print("X_valid_fe:", X_valid_fe.shape)
print("X_test_fe :", X_test_fe.shape)
pipeline_fe

Todas las verificaciones pasaron
X_train_fe: (9299486, 59)
X_valid_fe: (1614364, 59)
X_test_fe : (214200, 59)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('traducir_nombres', ...), ('imputar_mediana', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mappings,"{'ciudad': {'Адыгея': 'Adigueya', 'Балашиха': 'Balashija', 'Волжский': 'Volzhski', 'Вологда': 'Vologda', ...}, 'grupo_categoria': {'PC': 'Audifonos de PC', 'Аксессуары': 'Accesorios', 'Билеты (Цифра)': 'Boletos digitales', 'Доставка товара': 'Envios', ...}}"
,unknown_value,'Otro'
,variables,['venta_tienda_lag_1']
,imputation_method,'median'
Name,Type,Value
feature_names_in_,list,"['lag_1', 'lag_2', 'lag_3', 'ciudad', ...]"
imputer_dict_,dict,{'ve..._1': 0.23900206387043}


In [10]:
# Primeras filas del resultado
X_train_fe.head()

,lag_1,lag_2,lag_3,tienda_online,venta_tienda_lag_1,venta_producto_lag_1,meses_desde_primera_venta,mes,ciudad_Yaroslavl,ciudad_Moscu,ciudad_Kursk,ciudad_Krasnoyarsk,ciudad_Volzhski,ciudad_Voronezh,ciudad_Adigueya,ciudad_Balashija,ciudad_Yakutsk,ciudad_Kolomna,ciudad_Kaluga,ciudad_Zhukovski,ciudad_Kazan,ciudad_Tienda en linea,ciudad_Ufa,ciudad_Nizhni Novgorod,ciudad_Chejov,ciudad_Jimki,ciudad_Surgut,ciudad_Tiumen,ciudad_San Petersburgo,ciudad_Rostov del Don,ciudad_Omsk,ciudad_Samara,ciudad_Novosibirsk,ciudad_Sergiev Posad,ciudad_Vologda,ciudad_Almacen digital,ciudad_Ventas fuera de tienda,ciudad_Tomsk,ciudad_Mytishchi,grupo_categoria_Cine,grupo_categoria_Musica,grupo_categoria_Juegos de consola,grupo_categoria_Juegos de PC,grupo_categoria_Programas,grupo_categoria_Libros,grupo_categoria_Regalos,grupo_categoria_Accesorios,grupo_categoria_Consolas,grupo_categoria_Tarjetas de pago,grupo_categoria_Discos virgenes (unidad),grupo_categoria_Discos virgenes (paquete),grupo_categoria_Servicio interno,grupo_categoria_Audifonos de PC,grupo_categoria_Baterias,grupo_categoria_Envios,"grupo_categoria_Tarjetas de pago (cine, musica, juegos)",grupo_categoria_Boletos digitales,grupo_categoria_Juegos Android,grupo_categoria_Juegos MAC
0,-0.399026,-0.386636,-0.372799,0,-0.06016,-1.106133,-1.231101,-1.41966,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,-0.399026,-0.386636,-0.372799,0,-0.06016,-1.106133,-1.231101,-1.41966,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,-0.399026,-0.386636,-0.372799,0,-0.06016,-1.106133,-1.231101,-1.41966,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,-0.399026,-0.386636,-0.372799,0,-0.06016,-1.106133,-1.231101,-1.41966,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,-0.399026,-0.386636,-0.372799,0,-0.06016,-1.106133,-1.231101,-1.41966,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Interpretación del pipeline:**

- **Mapeo:** las 31 ciudades y los 20 grupos de categoría que aparecen en entrenamiento quedaron traducidos al
  español. Ningún valor quedó como `Otro`, es decir, el diccionario cubre todos los casos.
- **Imputación:** se rellenaron **453,373 vacíos** de `venta_tienda_lag_1` en entrenamiento con la mediana. Después
  de este paso no queda ninguno.
- **Codificación:** el One-Hot pasa de **10 a 59 columnas** (31 de ciudad + 20 de grupo, en lugar de las 2 columnas
  de texto).
- **Outliers:** el límite de `venta_producto_lag_1` es **3.83** unidades y se recortaron **92,698 filas** (el 1% más
  alto).
- **Yeo-Johnson:** el sesgo de la venta de la tienda bajó de 2.302 a 0.098 y el de la venta del producto de 4.336 a
  0.619. Los lags quedan con sesgo cercano a 2 porque la mayoría de sus valores son 0.
- **Escalado:** todas las variables escaladas quedan con media 0 y desviación 1.
- **Verificaciones (`assert`):** el pipeline completo da exactamente el mismo resultado que los pasos uno por uno, no
  deja vacíos en entrenamiento, validación ni test, y los tres sets tienen las mismas 59 columnas.

## Guardar el pipeline

Se guarda el pipeline **ajustado** en `models/pipeline_fe.pkl` con `joblib`. En el notebook 04 se carga y se clona
(como en clase) para entrenar cada modelo junto con el preprocesamiento.

Se comprueba que el archivo guardado, al cargarlo de nuevo, da el mismo resultado.

In [11]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ruta_pipeline = MODELS_DIR / "pipeline_fe.pkl"

joblib.dump(pipeline_fe, ruta_pipeline)

# Cargarlo de nuevo y comprobar que transforma igual
pipeline_cargado = joblib.load(ruta_pipeline)
X_test_check = pipeline_cargado.transform(X_test)
for columna in X_test_fe.columns:
    assert np.allclose(X_test_check[columna].astype(float), X_test_fe[columna].astype(float))

print("Pipeline guardado en:", ruta_pipeline)
print("Tamaño del archivo:", round(ruta_pipeline.stat().st_size / 1024**2, 1), "MB")

Pipeline guardado en: C:\Users\brady\OneDrive - Universidad Galileo\Documentos\PRODUCT DEVELOPMENT\Proyecto-1\models\pipeline_fe.pkl
Tamaño del archivo: 0.0 MB


**Resultado del notebook 03:**

- Se generó **`models/pipeline_fe.pkl`** con los seis pasos de ingeniería de características ya ajustados con el set
  de entrenamiento (meses 0 a 26).
- El archivo pesa unos **10 KB**, por eso el tamaño aparece como 0.0 MB. Es pequeño porque solo guarda lo aprendido
  en cada paso (la mediana, las categorías, el límite de outliers, los valores de Yeo-Johnson y del escalado), no los
  datos.
- Al cargarlo de nuevo, transforma el test exactamente igual, así que el archivo está listo para el notebook 04.
- Para abrir el `.pkl` en otro notebook o script, primero hay que importar `Mapper` desde
  `src/features/preprocesadores.py`.